# Pipeline CNN com tf.data e Data Augmentation

Este notebook documenta somente o que foi adicionado para a task de pipeline com augmentacao.

## Conceitos da task

**Data Augmentation**: tecnica que cria variacoes das imagens de treino em tempo de execucao (por exemplo, flip, rotacao e ajuste de contraste). O objetivo e aumentar a diversidade dos dados sem coletar novas amostras, reduzindo overfitting.

**Pipeline `tf.data`**: forma recomendada no TensorFlow para montar o fluxo de dados com etapas como `shuffle`, `batch`, `map` e `prefetch`. Isso melhora organizacao do treino e eficiencia de leitura/pre-processamento.

**Adequacao de tensores de entrada**: garante que os dados estejam no formato esperado pela CNN (`N, H, W, C`), com numero correto de canais e tipo numerico `float32`.

**Normalizacao por canal**: padroniza a escala de cada canal (ex.: `zscore`), ajustando os parametros no treino e reutilizando em validacao/teste para evitar vazamento de informacao.

Nesta task, combinamos esses quatro pontos para deixar o treinamento mais robusto e o protocolo experimental mais confiavel.

## O que foi feito de diferente

Comparando com `notebooks/cnn_preparacao_dados.ipynb`, este notebook adiciona: 

1. Pipeline `tf.data` pronto para treino (`shuffle -> batch -> map(augment) -> prefetch`).
2. Data augmentation no treino com `RandomFlip`, `RandomRotation` e `RandomContrast`.
3. Funcao unica para adequar tensores de entrada (`shape`, `data_format`, canais e `float32`).
4. Reuso explicito do normalizador do treino em validacao/teste dentro do pipeline.

Implementacao em `src/models/cnn_tf_data_pipeline.py`.

## Como usar no projeto

Fluxo pratico:

1. Tenha `X_train`, `y_train`, `X_val`, `y_val` prontos (por exemplo, saindo do `prepare_cnn_inputs`).
2. Chame `build_train_val_tf_data(...)` para criar `train_ds` e `val_ds`.
3. Treine com `model.fit(train_ds, validation_data=val_ds, ...)`.

Observacao: augmentacao fica ativa apenas no `train_ds`.

In [ ]:
from pathlib import Path
import sys
import numpy as np

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.cnn_tf_data_pipeline import build_train_val_tf_data


In [ ]:
# Exemplo sintetico para demonstrar uso
rng = np.random.default_rng(42)
x_train = rng.normal(100.0, 20.0, size=(64, 32, 32, 9)).astype(np.float32)
y_train = rng.integers(0, 2, size=(64,), dtype=np.int64)
x_val = rng.normal(100.0, 20.0, size=(16, 32, 32, 9)).astype(np.float32)
y_val = rng.integers(0, 2, size=(16,), dtype=np.int64)

out = build_train_val_tf_data(
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    batch_size=8,
    normalization='zscore',
    data_format='channels_last',
    target_channels=9,
    augment_train=True,
    seed=42,
)

train_ds = out['train_ds']
val_ds = out['val_ds']

xb, yb = next(iter(train_ds))
print('Batch treino:', xb.shape, yb.shape)
print('Input shape:', out['train_meta']['input_shape'])
print('Normalizacao:', out['train_meta']['normalization'])
print('Augmentacao treino:', out['train_meta']['augment'])


In [ ]:
# Integracao direta no treinamento
# history = model.fit(train_ds, validation_data=val_ds, epochs=30)


## Importancia disso

1. Reduz overfitting: augmentacao gera variacoes realistas e dificulta memorizacao do treino.
2. Melhora robustez: o modelo aprende a ser menos sensivel a orientacao e contraste.
3. Mantem avaliacao correta: validacao/teste sem augmentacao evitam metricas infladas.
4. Evita vazamento: normalizador ajustado no treino e reaplicado fora do treino.
5. Pipeline mais eficiente: `tf.data` usa `prefetch` e integra melhor com `model.fit`.

## Como isso afeta o modelo

Efeito esperado na pratica:

1. Treino pode ficar um pouco mais dificil no inicio (loss maior no comeco), porque os dados ficam mais variados.
2. Generalizacao tende a melhorar (gap treino-validacao menor).
3. Sensibilidade a mudancas de iluminacao/contraste tende a cair com `RandomContrast`.
4. Tempo por epoca pode aumentar um pouco, devido a augmentacao online.

Resumo: normalmente trocamos um pequeno custo de tempo por melhor desempenho fora do conjunto de treino.